# 🌲 Model Training and Evaluation - FloodGuard AI

This notebook documents the training pipeline of the **XGBoost** classifier used to predict urban flood risk zones. We load the geospatial synthetic datasets, perform a train-test split, initialize and train the multi-class model, evaluate it using standard classification metrics, and save the serialized model metadata.

## Objectives:
1. Define features (`elevation`, `slope`, `rainfall`, and rain lags) and split labels.
2. Partition dataset into Training (80%) and Testing (20%) sets.
3. Configure and fit the **XGBoost Classifier** model.
4. Assess model accuracy, F1-scores, and classification tables.
5. Save the final JSON model object.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("Libraries imported successfully!")

--- 
## 1. Load Data and Select Features

We load `flood_data.csv` and split it into predictor features ($X$) and target labels ($y$).

In [ ]:
data_path = "../data/flood_data.csv"
df = pd.read_csv(data_path)

# Select predictor features mapped to our train scripts
features = ['elevation', 'slope', 'rainfall', 'rainfall_lag_1', 'rainfall_lag_2']

X = df[features]
y = df['flood_risk']

print(f"Predictors: {features}")
print(f"Target class balance:\n{y.value_counts(normalize=True)*100}")

--- 
## 2. Train-Test Split

We split the data using an 80/20 partition to evaluate model performance on unseen data. We stratify by `y` to ensure consistent class distributions across partitions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training instances: {X_train.shape[0]}")
print(f"Testing instances: {X_test.shape[0]}")

--- 
## 3. Model Training (XGBoost)

We configure the multi-class gradient boosting classifier with: 
- `objective='multi:softmax'` to output discrete class labels (0, 1, or 2)
- `num_class=3` classes
- Hyperparameters: `n_estimators=100`, `max_depth=5`, and `learning_rate=0.1`

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=3,
    random_state=42
)

# Fit model on training partition
model.fit(X_train, y_train)
print("XGBoost model training complete!")

--- 
## 4. Evaluation

We score the test dataset and generate a classification report detailing Precision, Recall, and F1-score for each class.

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Global Accuracy: {accuracy * 100:.2f}%")
print(f"Weighted F1-Score: {f1:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'Medium Risk', 'High Risk']))

### Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=['Low', 'Medium', 'High'], yticklabels=['Low', 'Medium', 'High'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()

### Feature Importances
Understanding which features carry the highest predictive weight in decision boundaries is crucial for system explainability.

In [ ]:
importance = model.feature_importances_
feat_imp = pd.Series(importance, index=features).sort_values(ascending=True)

feat_imp.plot(kind='barh', color='teal')
plt.title('Feature Importances (Gain)')
plt.xlabel('Score')
plt.show()

--- 
## 5. Serialize the Trained Model

Finally, we save the trained XGBoost model structure to a JSON representation in the `models/` directory.

In [ ]:
save_dir = "../models"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "xgboost_flood_model.json")

model.save_model(save_path)
print(f"Model serialized successfully and saved to: {save_path}")